# 05 — Catalogue régional des lieux OpenStreetMap

**Objectif : livrer les établissements et lieux géographiques utiles recensés dans OSM**, avec une catégorie précise et une grande famille. Chaque ligne est un **objet OSM**, pas nécessairement un établissement distinct.

On réutilise les deux fichiers régionaux téléchargés et le contour IGN d'Auvergne-Rhône-Alpes, avec sa marge de 1 km. On ne télécharge rien dans ce notebook.

Ce catalogue balaie les catégories de lieux retenues dans `places_catalog.py` : commerces, services, santé, écoles, transport, tourisme, loisirs, patrimoine, relief, hydrographie et repères. Il conserve les tags OSM originaux et fournit une grande famille (`broad_category`) ainsi qu’une catégorie précise (`point_category`). Il ne représente pas chaque objet OSM (par exemple toutes les routes et tous les bâtiments) ni « tout ce qui existe sur le terrain ».

## 1. Préparer les chemins

Sélectionne le noyau **Python (IGN / OSM)**. Le code cherche la racine du dépôt pour fonctionner depuis VS Code, même si le dossier courant change. Le module `places_catalog.py` regroupe le traitement volumineux afin de garder ce notebook lisible.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display

current_directory = Path.cwd().resolve()
project_directory = next(
    (directory for directory in [current_directory, *current_directory.parents]
     if (directory / "regional-data" / "AGENTS.md").exists()),
    None,
)
if project_directory is None:
    raise FileNotFoundError("Open this notebook from the repository directory.")
module_directory = project_directory / "regional-data"
sys.path.insert(0, str(module_directory))
from places_catalog import build_catalog, verify_exports, classify_tags, CATEGORY_LABELS
output_directory = module_directory / "data" / "processed"

## 2. Comprendre la classification

Un tag OSM est une paire `clé=valeur`. Par exemple `amenity=pharmacy` identifie une pharmacie. La colonne `point_category` conserve cette catégorie exacte ; `broad_category` la range dans une famille, ici `health`.

Un objet peut porter plusieurs catégories : une station-service peut aussi avoir une boutique. Nous conservons **toutes les correspondances** dans `category_matches_json`. La catégorie principale suit un ordre explicite dans le module : équipement, santé, commerce, bureau, artisanat, tourisme, loisirs, sport, patrimoine, transport, secours, nature, eau, infrastructures, lieux habités et cols. Ce choix facilite les filtres ; il n'est pas une nomenclature officielle OSM.

Les valeurs non retenues incluent `no`, les états abandonnés/démolis et les constructions proposées. Les tags préfixés `disused:` ne sont pas sélectionnés seuls. Un établissement en cours de fermeture peut toutefois rester dans OSM : vérifie les tags conservés pour une analyse métier.

In [ ]:
examples = [
    {"amenity": "pharmacy", "name": "Example pharmacy"},
    {"shop": "bakery"},
    {"natural": "peak"},
    {"amenity": "fuel", "shop": "convenience"},
]
for tags in examples:
    print(tags, "->", classify_tags(tags))
display(pd.Series(CATEGORY_LABELS, name="label_fr").rename_axis("broad_category").to_frame())

## 3. Extraire les lieux de toute la région

Le fichier PBF contient des **points**, des **lignes** et des **relations**. Une relation multipolygone rassemble plusieurs morceaux d'un contour : le lecteur doit les assembler avant de reconstruire la surface.

Le traitement lit les deux sources, assemble les surfaces et garde les objets qui intersectent la région ou sa marge. Les objets communs aux deux fichiers sont fusionnés par `osm_uid` (`type/identifiant`). Nous gardons la version OSM la plus récente parmi celles disponibles.

La première exécution prend quelques minutes. Les suivantes réutilisent les exports uniquement si le code, les sources et le contour n'ont pas changé. Les erreurs de géométrie et les relations non reconstruites restent dans un fichier d'audit ; elles ne deviennent pas des objets fictifs.

In [ ]:
places, manifest = build_catalog(project_directory)
print(json.dumps({key: value for key, value in manifest.items() if key != "inputs"}, indent=2))

## 4. Explorer les résultats

**Attention au mot « point » :** `point_category` désigne le type de lieu. La géométrie réelle peut être une surface (parc), une ligne (rivière) ou un point (arrêt).

Dans le CSV, `longitude` et `latitude` sont des points représentatifs pour afficher les objets. Ils ne désignent pas nécessairement leur entrée et ne doivent pas remplacer leur géométrie pour mesurer des distances. Le GeoPackage conserve les formes originales.

In [ ]:
display(places[["osm_uid", "name", "broad_category", "point_category", "geometry_type", "inside_region"]].head(15))
display(places.groupby("broad_category").size().rename("mapped_object_count").sort_values(ascending=False).to_frame())
display(places["point_category"].value_counts().head(25).rename("mapped_object_count").to_frame())

## 5. Filtrer des lieux et lire leurs tags

Voici un exemple sur les commerces. Tu peux remplacer `retail` par `health` ou `mobility`.

`tags_json` préserve tous les attributs disponibles : adresse, horaires, capacité, opérateur, accès, etc. Une absence signifie **inconnu**, pas zéro. Pour un comptage, `countable=False` signale un point nommé situé dans une surface de même nom et même catégorie : c'est un doublon probable, conservé pour la traçabilité. Les autres doublons sémantiques restent possibles.

In [ ]:
retail_places = places.loc[places["broad_category"] == "retail"]
display(retail_places[["osm_uid", "name", "point_category", "longitude", "latitude"]].head(10))
if not retail_places.empty:
    display(json.loads(retail_places.iloc[0]["tags_json"]))
print(f"Potential point/area duplicates: {(~places['countable']).sum()}")

## 6. Vérifier et livrer

Les exports sont :

- `osm_places_catalog.gpkg` : formes originales, couche `places` ;
- `osm_places_catalog.csv` : tableau avec coordonnées d'affichage et tags JSON ;
- `osm_places_taxonomy.csv` : catégories réellement observées et libellés français des familles ;
- `osm_places_catalog_errors.csv` : problèmes et relations non reconstruites ;
- `osm_places_catalog_manifest.json` : sources, signature de calcul et volumes.

La marge d'1 km sert à ne pas ignorer un objet juste au-delà de la frontière. **Les fichiers sources restent des extraits régionaux** : une zone extérieure non incluse dans ces fichiers n'est pas couverte par magie. `inside_region` permet de filtrer les objets qui intersectent le contour régional, sans la marge.

Les résultats ne sont pas un registre officiel des établissements, et ne prouvent pas qu'un lieu est ouvert ou accessible au public. Consulter `access` et les tags originaux avant usage.

In [ ]:
verify_exports(project_directory, places)
errors = pd.read_csv(output_directory / "osm_places_catalog_errors.csv")
display(errors["reason"].value_counts().head(10).rename("issue_count").to_frame())
print("Catalog files are ready for reuse.")